## Assignment
@Author: **xx**

In [27]:
import pandas as pd
# Import the dataset as dataframe first.
dataset = pd.read_csv("dataset.tsv", sep="\t")
dataset.head(3)

,authors,title,genre,summary
0,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",young_adult,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...
1,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,fantasy,Harry Potter's life is miserable. His parents ...
2,Stephenie Meyer,"Twilight (Twilight, #1)",young_adult,About three things I was absolutely positive. ...


In [28]:
df = pd.DataFrame({'book_content': dataset.summary, 'genre':dataset.genre})

df.head(2)

,book_content,genre
0,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,young_adult
1,Harry Potter's life is miserable. His parents ...,fantasy


In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   book_content  4000 non-null   str  
 1   genre         4000 non-null   str  
dtypes: str(2)
memory usage: 62.6 KB


In [30]:
df = df.dropna()
df = df.drop_duplicates()

In [31]:
df.info()

<class 'pandas.DataFrame'>
Index: 3965 entries, 0 to 3999
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   book_content  3965 non-null   str  
 1   genre         3965 non-null   str  
dtypes: str(2)
memory usage: 92.9 KB


In [32]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

# text preprocessing
def preprocess_text(text):
    text = text.lower()
    space_version_text = re.sub(r'[^\w\s]+', ' ', text.lower())
    tokens = word_tokenize(space_version_text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [ps.stem(word) for word in tokens]
    return ' '.join(tokens)

# Apply preprocessing to each book
df['book_content'] = df['book_content'].apply(preprocess_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/carside/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/carside/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/carside/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [33]:
df['book_content'].head(2)

0    win mean fame fortun lose mean certain death h...
1    harri potter life miser parent dead stuck hear...
Name: book_content, dtype: str

### Model Training and Evaluation

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


# Convert text data into TF-IDF weights
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['book_content'])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, df['genre'], test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

(3172, 24737) (793, 24737)


__Model Training__

In [36]:
from sklearn.naive_bayes import BernoulliNB

# Train the Bernoulli Naive Bayes model
bnb = BernoulliNB()
bnb.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"binarize binarize: float or None, default=0.0Threshold for binarizing (mapping to booleans) of sample features.If None, input is presumed to already consist of binary vectors.",0.0
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


__Model Evaluation__

In [37]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = bnb.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


0.6279949558638083
              precision    recall  f1-score   support

    classics       0.77      0.79      0.78       178
     fantasy       0.47      0.90      0.62       225
     romance       0.85      0.47      0.60       143
    thriller       0.96      0.44      0.61       115
 young_adult       0.76      0.28      0.41       132

    accuracy                           0.63       793
   macro avg       0.76      0.58      0.60       793
weighted avg       0.72      0.63      0.62       793



In [38]:
from sklearn.naive_bayes import MultinomialNB
multinomial_nb = MultinomialNB()
multinomial_nb.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [39]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = multinomial_nb.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


0.4905422446406053
              precision    recall  f1-score   support

    classics       0.82      0.54      0.66       178
     fantasy       0.37      0.97      0.53       225
     romance       0.95      0.29      0.45       143
    thriller       0.94      0.15      0.26       115
 young_adult       0.93      0.11      0.19       132

    accuracy                           0.49       793
   macro avg       0.80      0.41      0.42       793
weighted avg       0.75      0.49      0.45       793

